In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import re
import datetime
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示', 'スタートタイミング']

In [3]:
k_files = glob.glob("../csv/K*")
b_files = glob.glob("../csv/B*")

In [4]:
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp

In [5]:
print("Concat K-files")
kdf = concat_files(k_files)
print("Concat B-files")
bdf = concat_files(b_files)

100%|██████████| 843/843 [01:11<00:00, 11.75it/s]


In [6]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 712541 entries, 0 to 712540
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   艇番         712541 non-null  int64  
 1   選手登番       712541 non-null  int64  
 2   選手名        712541 non-null  object 
 3   年齢         712541 non-null  int64  
 4   支部         712541 non-null  object 
 5   体重         712541 non-null  int64  
 6   級別         712541 non-null  object 
 7   全国勝率       712541 non-null  float64
 8   全国2連率      712541 non-null  float64
 9   当地勝率       712541 non-null  float64
 10  当地2連率      712541 non-null  float64
 11  モーター2連率    712541 non-null  float64
 12  ボート2連率     712541 non-null  float64
 13  RaceID     712541 non-null  object 
 14  場所         712541 non-null  int64  
 15  R          712541 non-null  object 
 16  着          712541 non-null  int64  
 17  展示         712541 non-null  float64
 18  スタートタイミング  712541 non-null  float64
dtypes: float64(8), int64(6)

In [8]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid


In [9]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

In [10]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 712541/712541 [00:04<00:00, 144375.12it/s]


In [11]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 712541 entries, 0 to 712540
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   艇番         712541 non-null  int64  
 1   選手登番       712541 non-null  int64  
 2   選手名        712541 non-null  object 
 3   年齢         712541 non-null  int64  
 4   支部         712541 non-null  int64  
 5   体重         712541 non-null  int64  
 6   級別         712541 non-null  int64  
 7   全国勝率       712541 non-null  float64
 8   全国2連率      712541 non-null  float64
 9   当地勝率       712541 non-null  float64
 10  当地2連率      712541 non-null  float64
 11  モーター2連率    712541 non-null  float64
 12  ボート2連率     712541 non-null  float64
 13  RaceID     712541 non-null  object 
 14  場所         712541 non-null  int64  
 15  R          712541 non-null  int64  
 16  着          712541 non-null  int64  
 17  展示         712541 non-null  float64
 18  スタートタイミング  712541 non-null  float64
dtypes: float64(8), int64(9)

In [12]:
os.makedirs('../train_csv',exist_ok=True)
df_encoded.to_csv('../train_csv/train.csv')